In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.integrate import cumulative_trapezoid

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("switch colab runtime to GPU first")
device = "cuda"

N_TRAIN = 10000
N_TEST = 1000
M_SENSORS = 100
P_POINTS = 1
GRF_LENGTH_SCALE = 0.2

PINN_ITERS = 40000
DON_ITERS = 40000
PI_ITERS = 40000
LR = 1e-3
LAMBDA_PHYS = 1.0
BATCH = 10000

N_PINN_COLLOC = 100
N_PI_PHYS = 100

WIDTH = 40
DEPTH = 2

RUN_TAG = "linear_ode_v2"

def sample_grf(n_samples, n_points, length_scale):
    x = np.linspace(0, 1, n_points)
    xi, xj = np.meshgrid(x, x)
    K = np.exp(-0.5 * ((xi - xj) / length_scale) ** 2) + 1e-10 * np.eye(n_points)
    L = np.linalg.cholesky(K)
    z = np.random.randn(n_samples, n_points)
    return z @ L.T

x_sensors = np.linspace(0, 1, M_SENSORS)

def make_dataset(n_samples):
    u_at_sensors = sample_grf(n_samples, M_SENSORS, GRF_LENGTH_SCALE).astype(np.float32)
    s_full = cumulative_trapezoid(u_at_sensors, x_sensors, initial=0, axis=1).astype(np.float32)
    y_query = np.random.rand(n_samples, P_POINTS).astype(np.float32)
    s_query = np.empty((n_samples, P_POINTS), dtype=np.float32)
    for i in range(n_samples):
        s_query[i] = np.interp(y_query[i], x_sensors, s_full[i])
    return u_at_sensors, y_query, s_query, s_full

print("building datasets")
u_train, y_train, s_train, _ = make_dataset(N_TRAIN)
u_test, y_test, s_test, s_test_full = make_dataset(N_TEST)

def mlp(sizes, act):
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(torch.nn.Linear(sizes[i], sizes[i + 1]))
        if i < len(sizes) - 2:
            layers.append(act())
    return torch.nn.Sequential(*layers)

class PINN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = mlp([1] + [WIDTH] * DEPTH + [1], torch.nn.Tanh)

    def forward(self, x):
        return self.net(x)

class DeepONet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.branch = mlp([M_SENSORS] + [WIDTH] * DEPTH, torch.nn.Tanh)
        self.trunk = mlp([1] + [WIDTH] * DEPTH, torch.nn.Tanh)
        self.bias = torch.nn.Parameter(torch.zeros(1))

    def forward(self, u, y):
        b = self.branch(u)
        t = self.trunk(y)
        return (b * t).sum(dim=1, keepdim=True) + self.bias

class PIDeepONet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.branch = mlp([M_SENSORS] + [WIDTH] * DEPTH, torch.nn.Tanh)
        self.trunk = mlp([1] + [WIDTH] * DEPTH, torch.nn.Tanh)
        self.bias = torch.nn.Parameter(torch.zeros(1))

    def forward(self, u, y):
        b = self.branch(u)
        t = self.trunk(y)
        return (b * t).sum(dim=1, keepdim=True) + self.bias

def rel_l2(pred, true):
    return np.linalg.norm(pred - true) / np.linalg.norm(true)

print("\n" + "=" * 60)
print("model 1/3: PINN")
print("=" * 60)
print("PINN trains on ONE specific u(x), not the operator")
print("using the first test sample as the fixed forcing")

def u_interp_torch(x_query, u_vals):
    x_q_np = x_query.detach().cpu().numpy().flatten()
    u_at_q = np.interp(x_q_np, x_sensors, u_vals[0])
    return torch.tensor(u_at_q.reshape(-1, 1), dtype=torch.float32, device=device)

pinn_net = PINN().to(device)
pinn_opt = torch.optim.Adam(pinn_net.parameters(), lr=LR)

x_ic = torch.zeros(1, 1, device=device)

for step in range(PINN_ITERS + 1):
    x_c = torch.rand(N_PINN_COLLOC, 1, device=device, requires_grad=True)
    u_at_c = u_interp_torch(x_c, u_test[0:1])

    s_pred = pinn_net(x_c)
    ds_dx = torch.autograd.grad(s_pred, x_c, grad_outputs=torch.ones_like(s_pred), create_graph=True)[0]
    loss_phys = ((ds_dx - u_at_c) ** 2).mean()

    s_ic = pinn_net(x_ic)
    loss_ic = (s_ic ** 2).mean()

    loss = loss_phys + loss_ic
    pinn_opt.zero_grad()
    loss.backward()
    pinn_opt.step()

    if step % 5000 == 0:
        print(f"pinn step {step:6d} | phys {loss_phys.item():.3e} | ic {loss_ic.item():.3e}")

with torch.no_grad():
    x_eval = torch.tensor(x_sensors.reshape(-1, 1).astype(np.float32), device=device)
    s_pinn_pred = pinn_net(x_eval).cpu().numpy().flatten()
s_pinn_true = cumulative_trapezoid(u_test[0:1], x_sensors, initial=0, axis=1)[0]
pinn_error = rel_l2(s_pinn_pred, s_pinn_true)
print(f"PINN rel L2 error on the one flow it was trained on: {pinn_error:.4%}")

print("\n" + "=" * 60)
print("model 2/3: DeepONet (data only)")
print("=" * 60)

don = DeepONet().to(device)
don_opt = torch.optim.Adam(don.parameters(), lr=LR)

u_train_t = torch.tensor(u_train, device=device)
y_train_t = torch.tensor(y_train, device=device)
s_train_t = torch.tensor(s_train, device=device)

for step in range(DON_ITERS + 1):
    idx = torch.randint(0, N_TRAIN, (BATCH,))
    u_b = u_train_t[idx]
    y_b = y_train_t[idx]
    s_b = s_train_t[idx]

    pred = don(u_b, y_b)
    loss = ((pred - s_b) ** 2).mean()

    don_opt.zero_grad()
    loss.backward()
    don_opt.step()

    if step % 5000 == 0:
        print(f"don step {step:6d} | data {loss.item():.3e}")

with torch.no_grad():
    u_test_t = torch.tensor(u_test, device=device)
    don_preds = np.zeros((N_TEST, M_SENSORS))
    for j in range(M_SENSORS):
        y_j = torch.full((N_TEST, 1), x_sensors[j], device=device, dtype=torch.float32)
        don_preds[:, j] = don(u_test_t, y_j).cpu().numpy().flatten()

don_errors = [rel_l2(don_preds[i], s_test_full[i]) for i in range(N_TEST)]
don_mean_error = np.mean(don_errors)
don_std_error = np.std(don_errors)
print(f"DeepONet mean rel L2 error over {N_TEST} test samples: {don_mean_error:.4%} +/- {don_std_error:.4%}")

print("\n" + "=" * 60)
print("model 3/3: PI-DeepONet (data + physics)")
print("=" * 60)

pinet = PIDeepONet().to(device)
pi_opt = torch.optim.Adam(pinet.parameters(), lr=LR)

for step in range(PI_ITERS + 1):
    idx = torch.randint(0, N_TRAIN, (BATCH,))
    u_b = u_train_t[idx]
    y_b = y_train_t[idx]
    s_b = s_train_t[idx]

    pred = pinet(u_b, y_b)
    loss_data = ((pred - s_b) ** 2).mean()

    idx_p = torch.randint(0, N_TRAIN, (BATCH,))
    u_p = u_train_t[idx_p]
    y_p = torch.rand(BATCH, 1, device=device, requires_grad=True)
    s_p = pinet(u_p, y_p)
    ds_dy = torch.autograd.grad(s_p, y_p, grad_outputs=torch.ones_like(s_p), create_graph=True)[0]

    y_p_np = y_p.detach().cpu().numpy().flatten()
    idx_p_np = idx_p.cpu().numpy()
    u_p_np = u_train[idx_p_np]
    idx_left = np.clip(np.searchsorted(x_sensors, y_p_np) - 1, 0, M_SENSORS - 2)
    x_left = x_sensors[idx_left]
    x_right = x_sensors[idx_left + 1]
    alpha = ((y_p_np - x_left) / (x_right - x_left)).astype(np.float32)
    u_left = u_p_np[np.arange(BATCH), idx_left]
    u_right = u_p_np[np.arange(BATCH), idx_left + 1]
    u_at_y_np = u_left + alpha * (u_right - u_left)
    u_at_y = torch.tensor(u_at_y_np.reshape(-1, 1), device=device)

    loss_phys = ((ds_dy - u_at_y) ** 2).mean()

    y_ic = torch.zeros(BATCH, 1, device=device)
    s_ic = pinet(u_p, y_ic)
    loss_ic = (s_ic ** 2).mean()

    loss = loss_data + LAMBDA_PHYS * loss_phys + loss_ic

    pi_opt.zero_grad()
    loss.backward()
    pi_opt.step()

    if step % 5000 == 0:
        print(f"pi  step {step:6d} | data {loss_data.item():.3e} | phys {loss_phys.item():.3e} | ic {loss_ic.item():.3e}")

with torch.no_grad():
    pi_preds = np.zeros((N_TEST, M_SENSORS))
    for j in range(M_SENSORS):
        y_j = torch.full((N_TEST, 1), x_sensors[j], device=device, dtype=torch.float32)
        pi_preds[:, j] = pinet(u_test_t, y_j).cpu().numpy().flatten()

pi_errors = [rel_l2(pi_preds[i], s_test_full[i]) for i in range(N_TEST)]
pi_mean_error = np.mean(pi_errors)
pi_std_error = np.std(pi_errors)
print(f"PI-DeepONet mean rel L2 error over {N_TEST} test samples: {pi_mean_error:.4%} +/- {pi_std_error:.4%}")

print("\n" + "=" * 70)
print(f"  FINAL METRICS: {RUN_TAG}")
print("=" * 70)
print(f"{'model':<22}{'rel L2 error':<24}{'note':<30}")
print(f"{'PINN':<22}{f'{pinn_error:.4%}':<24}{'one specific u(x)':<30}")
print(f"{'DeepONet':<22}{f'{don_mean_error:.4%} +/- {don_std_error:.4%}':<24}{f'operator, {N_TEST} samples':<30}")
print(f"{'PI-DeepONet':<22}{f'{pi_mean_error:.4%} +/- {pi_std_error:.4%}':<24}{f'operator, {N_TEST} samples':<30}")
print(f"{'Wang et al reported':<22}{'0.33% +/- 0.32%':<24}{'PI-DeepONet, Table 1':<30}")
print("=" * 70)

idx_show = 0
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(x_sensors, u_test[idx_show], label="u(x) forcing", color="black")
plt.xlabel("x")
plt.ylabel("u(x)")
plt.title("input forcing function u(x)")
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(x_sensors, s_test_full[idx_show], label="true s(x)", color="black", linewidth=2)
plt.plot(x_sensors, don_preds[idx_show], "--", label="DeepONet", color="tab:red")
plt.plot(x_sensors, pi_preds[idx_show], "--", label="PI-DeepONet", color="tab:blue")
plt.xlabel("x")
plt.ylabel("s(x)")
plt.title(f"solution s(x) for test sample {idx_show}")
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.savefig(f"{RUN_TAG}_prediction.png", dpi=200, bbox_inches="tight")
plt.show()

rows = [
    ["PINN", f"{pinn_error:.4%}", "trained on one specific u(x)"],
    ["DeepONet", f"{don_mean_error:.4%} +/- {don_std_error:.4%}", f"operator, {N_TEST} test samples"],
    ["PI-DeepONet", f"{pi_mean_error:.4%} +/- {pi_std_error:.4%}", f"operator, {N_TEST} test samples"],
    ["Wang et al (paper)", "0.33% +/- 0.32%", "reported PI-DeepONet"],
]

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.axis("off")
tbl = ax.table(cellText=rows,
               colLabels=["model", "rel L2 error", "note"],
               loc="center", cellLoc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.6)
ax.set_title("linear ODE: ds/dx = u(x), s(0) = 0", fontsize=11, pad=10)
plt.savefig(f"{RUN_TAG}_metrics_table.png", dpi=200, bbox_inches="tight")
plt.show()

np.save("errs_linear_ode.npy", {"pinn": pinn_error, "don_mean": don_mean_error, "pi_mean": pi_mean_error})